Этот ноутбук посвящён XGBoost, LightGBM, CatBoost.  
Все эти три алгоритма - реализации градиентного бустинга над решающими деревьями. Они решают одну и ту же задачу, но отличаются способом построения деревьев, вычислением сплитов, работой с признаками и регуляризацией

XGBoost и LightGBM существенно различаются способом построения деревьев: XGBoost строит деревья по уровням, а в LightGBM - берём "лучший" лист(лист с максимальным gain, gain = loss до сплита - loss после сплита с поправкой на регуларизацию) и расщепляем именно его  
При одинаковом количестве листьев LightGBM может получить более сложную структуру и быстрее переобучаться

CatBoost строит деревья по уровням, но сплит использует ко всему уровню!  
А ещё CatBoost поддерживает категориальные фичи из коробки, поскольку специально под работу с категориальными фичами и оптимизировался. Он использует ordered target statistics и ordered boosting, что позволяет использовать информацию о target для кодирования категорий, снижая риск target leakage и prediction shift

LightGBM отличается скоростью работы, поскольку использует ряд оптимизаций, а также histogram-based алгоритм(реализация такого есть и в sklearn.ensemble.HistGradientBoosting)  
Непрерывные значения признаков распределяются по корзинам(bins), а поиск сплитов выполняется по гистограммам, а не по всем уникальным значениям

Какой бустинг лучше выбрать?  
Если много категориальных фичей - CatBoost  
Если важна скорость, а табличный датасет большой - LightGBM  
Если нужен сильный универсальный бейзлайн - XGBoost

In [2]:
%pip install xgboost lightgbm catboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 4.2 MB/s eta 0:00:24
   - -------------------------------------- 2.6/101.7 MB 7.2 MB/s eta 0:00:14
   - -------------------------------------- 5.0/101.7 MB 8.9 MB/s eta 0:00:11
   -- ------------------------------------- 7.6/101.7 MB 9.8 MB/s eta 0:00:10
   ---- ----------------------------------- 10.2/101.7 MB 10.3 MB/s eta 0:00:09
   ---- ----------------------------------- 12.6/101.7 MB 10.5 MB/s eta 0:00:09
   ----- ---------------------------------- 15.2/101.7 MB 10.6 MB/s eta 0:00:09
   ------ --------------------------------- 17.6/101.7 MB 10.8 MB/s eta 0:00:08
   ------- -------------------------------- 19.9/101.7 MB 10.9 MB/s eta 0:00:08
   -------- ------------------------------- 22.3/101.7 MB 10.8 MB/s eta 0:00:08
   --------- ------------------------------ 24.6/101.7 MB 10.9 MB/

In [9]:
import xgboost as xgb
import lightgbm as lgbm
import catboost
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import time
from sklearn.metrics import f1_score

In [22]:
cb = catboost.CatBoostClassifier()
lightgbm = lgbm.LGBMClassifier()
xgboost = xgb.XGBClassifier(enable_categorical=True)

In [13]:
df = pd.read_csv('../datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [14]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [17]:
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn']
y = np.where(y == 'Yes', 1, 0)
for col in X.select_dtypes(include='str').columns:
    X[col] = X[col].replace(r'^\s*$', np.nan, regex=True)
    try:
        X[col] = pd.to_numeric(X[col])
    except:
        X[col] = X[col].astype('category')
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)


Первым затестим катбуст, потому что у этого датасета есть один прикол


In [19]:
categorial_features = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
                      'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',  'PaperlessBilling', 'PaymentMethod']
cb.fit(X_train, y_train, cat_features=categorial_features)

y_cat = cb.predict(X_test)
f1 = f1_score(y_test, y_cat)
print(f1)

Learning rate set to 0.020969
0:	learn: 0.6792494	total: 228ms	remaining: 3m 48s
1:	learn: 0.6661233	total: 279ms	remaining: 2m 19s
2:	learn: 0.6538600	total: 330ms	remaining: 1m 49s
3:	learn: 0.6418297	total: 379ms	remaining: 1m 34s
4:	learn: 0.6316056	total: 427ms	remaining: 1m 25s
5:	learn: 0.6216125	total: 478ms	remaining: 1m 19s
6:	learn: 0.6141799	total: 496ms	remaining: 1m 10s
7:	learn: 0.6048321	total: 557ms	remaining: 1m 9s
8:	learn: 0.5961369	total: 610ms	remaining: 1m 7s
9:	learn: 0.5887387	total: 649ms	remaining: 1m 4s
10:	learn: 0.5806614	total: 715ms	remaining: 1m 4s
11:	learn: 0.5737389	total: 767ms	remaining: 1m 3s
12:	learn: 0.5664152	total: 833ms	remaining: 1m 3s
13:	learn: 0.5598259	total: 868ms	remaining: 1m 1s
14:	learn: 0.5546978	total: 899ms	remaining: 59s
15:	learn: 0.5484719	total: 940ms	remaining: 57.8s
16:	learn: 0.5428827	total: 972ms	remaining: 56.2s
17:	learn: 0.5373370	total: 1.01s	remaining: 55.1s
18:	learn: 0.5321136	total: 1.05s	remaining: 54.3s
19:	le

In [21]:
start = time.perf_counter()
lightgbm.fit(X_train, y_train)
end = time.perf_counter() - start
y_light = lightgbm.predict(X_test)
f1 = f1_score(y_test, y_light)
print(f1)
print(end)

[LightGBM] [Info] Number of positive: 1402, number of negative: 3880
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000579 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 641
[LightGBM] [Info] Number of data points in the train set: 5282, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265430 -> initscore=-1.017935
[LightGBM] [Info] Start training from score -1.017935
0.5673758865248227
0.09313569986261427


LightGBM обучился меньше чем за десятую секунды, когда кк катбуст обучался около минуты

In [23]:
start = time.perf_counter()
xgboost.fit(X_train, y_train)
end = time.perf_counter() - start
f1 = f1_score(y_test, y_xg)
print(f1)
print(end)

0.5225653206650831
0.44082899997010827
